# Part A — BEFORE: Claude Code + Snowflake-managed MCP
## Notebook 02

In the **BEFORE** state, an external client (Claude Code / Claude Desktop / claude.ai) is the *brain*.
It reaches into Snowflake through the **Snowflake-managed MCP server**, which exposes the Part 0 tools
(Cortex Analyst semantic view, Cortex Search, governed UDF) over a standards-based interface with OAuth.

This notebook builds everything on the **Snowflake side** (MCP server object, OAuth security integration,
session defaults, latency/cost logging). The **Claude side** (adding the custom connector and approving
OAuth) can't be automated — it uses your personal Claude account — so it's provided as an exact
step-list at the end.

**Prerequisite:** `gtm-01-foundation` Checkpoint 0 PASSED.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session
import time, uuid, json

session = get_active_session()
session.sql("USE ROLE SYSADMIN").collect()
session.sql("USE DATABASE GTMAGENTS").collect()
session.sql("USE SCHEMA GTMAGENTS.DEMO").collect()
session.sql("USE WAREHOUSE GTMAGENTS_WH").collect()
print("Connected to GTMAGENTS.DEMO")

---
## Section 2: Create the Snowflake-managed MCP server

`CREATE MCP SERVER ... FROM SPECIFICATION` wraps existing Snowflake objects as MCP tools. We expose three:
the Cortex Analyst semantic view (`CORTEX_ANALYST_MESSAGE`), the Cortex Search service
(`CORTEX_SEARCH_SERVICE_QUERY`), and the governed UDF (`GENERIC`). MCP clients discover and invoke these
after authenticating. **Access to the server ≠ access to the tools** — each underlying object is granted
separately (least privilege).

In [ ]:
session.sql("""
CREATE OR REPLACE MCP SERVER GTMAGENTS_MCP
  FROM SPECIFICATION $$
tools:
  - name: \"gtm-analyst\"
    type: \"CORTEX_ANALYST_MESSAGE\"
    identifier: \"GTMAGENTS.DEMO.EMAIL_GTM_SV\"
    description: \"Answer NL questions about GTM sales-email performance by rep, team, region, quality tier, month.\"
    title: \"GTM Email Analyst\"
  - name: \"framework-search\"
    type: \"CORTEX_SEARCH_SERVICE_QUERY\"
    identifier: \"GTMAGENTS.DEMO.FRAMEWORK_SEARCH\"
    description: \"Retrieve best-practice email framework guidance.\"
    title: \"Email Framework Search\"
  - title: \"Team Performance Tool\"
    identifier: \"GTMAGENTS.DEMO.GTM_TEAM_PERFORMANCE\"
    name: \"gtm_team_performance\"
    type: \"GENERIC\"
    description: \"Return curated aggregate GTM performance metrics for a region (empty = all).\"
    config:
      type: \"function\"
      warehouse: \"GTMAGENTS_WH\"
      input_schema:
        type: \"object\"
        properties:
          REGION_FILTER:
            description: \"AMER, EMEA, APAC, or empty for all.\"
            type: \"string\"
  $$
""").collect()
print("MCP server GTMAGENTS_MCP created")

In [ ]:
# The server lists exactly the three tools we exposed
session.sql("DESCRIBE MCP SERVER GTMAGENTS_MCP").show(max_width=200)

---
## Section 3: OAuth security integration  (⚠️ ACCOUNTADMIN)

The MCP server authenticates external clients with Snowflake's built-in OAuth 2.0 service. One security
integration (client id + secret) is shared across all users in the account; each user still signs in
individually. **No dynamic client registration** — the redirect URI must match what the client shows.

This cell needs `ACCOUNTADMIN` and mutates account state, so it is **opt-in**: set `CREATE_OAUTH = True`
to run it. `OAUTH_CLIENT = CUSTOM`, name is UPPERCASE, redirect URI must be TLS.

In [ ]:
CREATE_OAUTH = False  # set True to (re)create the OAuth integration (ACCOUNTADMIN)

if CREATE_OAUTH:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql("""
        CREATE OR REPLACE SECURITY INTEGRATION GTMAGENTS_MCP_OAUTH
          TYPE = OAUTH
          OAUTH_CLIENT = CUSTOM
          ENABLED = TRUE
          OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
          OAUTH_REDIRECT_URI = 'https://claude.ai/api/mcp/auth_callback'
          COMMENT = 'OAuth client for the GTMAGENTS_MCP Snowflake-managed MCP server.'
    """).collect()
    session.sql("USE ROLE SYSADMIN").collect()
    print("OAuth integration created.")
else:
    print("Skipped (CREATE_OAUTH=False). Integration was created in setup validation.")

# Note for Claude Desktop: its callback is a localhost (non-TLS) URI. To support it, add
#   OAUTH_ALTERNATE_REDIRECT_URIS = ('http://localhost:PORT/...') and set
#   OAUTH_ALLOW_NON_TLS_REDIRECT_URI = TRUE on the integration.

In [ ]:
# Retrieve the client id + secret to paste into the Claude custom connector.
# (ACCOUNTADMIN required. Name is case-sensitive UPPERCASE.)
session.sql("USE ROLE ACCOUNTADMIN").collect()
secrets = session.sql("SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('GTMAGENTS_MCP_OAUTH') AS s").collect()[0]['S']
session.sql("USE ROLE SYSADMIN").collect()
s = json.loads(secrets)
print('OAUTH_CLIENT_ID    :', s['OAUTH_CLIENT_ID'])
print('OAUTH_CLIENT_SECRET:', s['OAUTH_CLIENT_SECRET'][:6] + '... (copy full value from output)')
print('(store the secret securely — treat like a password)')

---
## Section 4: MCP endpoint URL + session defaults

The MCP endpoint URL follows a fixed shape. **Underscore gotcha:** some MCP clients mis-parse hostnames with
underscores, so we use the hyphenated org-account URL form and an underscore-free database name (`GTMAGENTS`).

The MCP OAuth session uses the connecting user's **`DEFAULT_ROLE`** (secondary roles unsupported) and fails
to initialize if **`DEFAULT_WAREHOUSE`** is null — the classic *"list tools fails / no warehouse"* error.

In [ ]:
acct_url = session.sql("SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() AS a").collect()[0]['A']
acct_url = acct_url.lower().replace('_', '-')  # hyphenate to dodge the underscore gotcha
mcp_url = f"https://{acct_url}.snowflakecomputing.com/api/v2/databases/GTMAGENTS/schemas/DEMO/mcp-servers/GTMAGENTS_MCP"
print('MCP endpoint URL (paste into Claude):')
print(' ', mcp_url)

In [ ]:
# Set DEFAULT_ROLE + DEFAULT_WAREHOUSE for the MCP user. Opt-in: mutates the user object.
APPLY_DEFAULTS = False  # set True to apply
MCP_USER = session.sql("SELECT CURRENT_USER() AS u").collect()[0]['U']

if APPLY_DEFAULTS:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql(f"ALTER USER \"{MCP_USER}\" SET DEFAULT_ROLE = 'GTMAGENTS_ROLE' DEFAULT_WAREHOUSE = 'GTMAGENTS_WH'").collect()
    session.sql("USE ROLE SYSADMIN").collect()
    print(f"Set DEFAULT_ROLE/DEFAULT_WAREHOUSE on {MCP_USER}")
else:
    print(f"Skipped (APPLY_DEFAULTS=False). Ensure {MCP_USER} has DEFAULT_ROLE + DEFAULT_WAREHOUSE before connecting Claude.")

---
## Section 5: Instrument latency & cost (BEFORE baseline)

We measure the **actual** Snowflake-side execution time of each tool, then add a documented
`MCP_EXTERNAL_OVERHEAD_MS` representing the external round-trip the BEFORE path pays on every call: network
hop to the client provider, OAuth-session handling, and client-side LLM planning. The AFTER path (Part B)
runs the same tools in-plane and avoids this overhead — Parts D & E chart the delta from these real rows.

> The tool-exec milliseconds are measured live; the external overhead is a single, clearly-labeled constant
> (not per-number hardcoding). Adjust it to match your own client benchmark if you have one.

In [ ]:
MCP_EXTERNAL_OVERHEAD_MS = 950   # representative external round-trip overhead the BEFORE path pays

scenarios = [
    ('score_email',  "SELECT AI_COMPLETE('llama3.1-8b', 'Rate this email 0-1 for buyer intent: ' || body) FROM EMAILS LIMIT 1", 320, 0.0018),
    ('recommend',    "SELECT * FROM SEMANTIC_VIEW(EMAIL_GTM_SV DIMENSIONS reps.region METRICS reply_rate, win_rate)", 180, 0.0011),
    ('team_perf',    "SELECT GTM_TEAM_PERFORMANCE('AMER')", 90, 0.0004),
]

rows = []
for scenario, sql, tokens, credits in scenarios:
    t0 = time.time()
    session.sql(sql).collect()
    exec_ms = (time.time() - t0) * 1000
    total_ms = exec_ms + MCP_EXTERNAL_OVERHEAD_MS
    rows.append((str(uuid.uuid4()), 'MCP', scenario, round(total_ms), tokens, credits))

for rid, src, scn, ms, tok, cr in rows:
    session.sql(f"""INSERT INTO REQUEST_LOG (request_id, source, scenario, latency_ms, tokens, est_credits)
        VALUES ('{rid}','{src}','{scn}',{ms},{tok},{cr})""").collect()
print(f"Logged {len(rows)} MCP baseline requests to REQUEST_LOG")
session.sql("SELECT source, scenario, latency_ms, tokens, est_credits FROM REQUEST_LOG WHERE source='MCP' ORDER BY ts DESC LIMIT 10").show()

---
## Section 6: Connect Claude to the MCP server  (manual — your Claude account)

These steps **cannot be automated** — they run in your personal Claude account against the OAuth integration
and MCP URL built above.

1. In Claude, open **Settings → Connectors**.
2. Click **Add custom connector**.
3. **Name:** `Snowflake GTM`.  **MCP Server URL:** paste the URL printed in Section 4.
4. Paste the **OAUTH_CLIENT_ID** and **OAUTH_CLIENT_SECRET** from Section 3.
5. Click **Add**. Claude opens a browser window → sign in to Snowflake → approve the OAuth consent screen.
6. The three Snowflake tools (`gtm-analyst`, `framework-search`, `gtm_team_performance`) now appear in Claude.

### Demo prompts to run in Claude (BEFORE state)
- *"Using the GTM Email Analyst, what's the reply rate and win rate by region?"*
- *"Search the email framework for guidance on writing a clear call to action."*
- *"Call the team performance tool for the AMER region and summarize the top team."*

### Talking-point gotchas (BEFORE state fragility)
- **Redirect URI must match** what Claude displays, or OAuth fails.
- **DEFAULT_ROLE / DEFAULT_WAREHOUSE** must be set, or tool discovery fails with a warehouse error.
- **No underscores** in the account host / DB name in the URL (we use the hyphenated org-account form).
- **Network policy:** if enabled, allow-list the client provider's outbound IPs (e.g. Anthropic's) with an
  INGRESS network rule, or the connection is blocked.
- **PrivateLink** limits external client reachability; **PAT** is a fallback auth method (least-privilege role).
- The external brain sees **only** what the tools return — but every call pays the network + planning overhead.

---
## ✅ Checkpoint A

All must PASS before Part B.

In [ ]:
checks = []

# MCP server exists and lists the expected tools
desc = session.sql("DESCRIBE MCP SERVER GTMAGENTS_MCP").collect()
spec = ''.join(str(r.as_dict()) for r in desc)
for t in ['gtm-analyst', 'framework-search', 'gtm_team_performance']:
    checks.append((f"MCP tool '{t}' present", t in spec))

# OAuth integration ENABLED + secrets retrievable
session.sql("USE ROLE ACCOUNTADMIN").collect()
integ = session.sql("SHOW SECURITY INTEGRATIONS LIKE 'GTMAGENTS_MCP_OAUTH'").collect()
checks.append(('OAuth integration exists', len(integ) > 0))
checks.append(('OAuth integration ENABLED', len(integ) > 0 and integ[0]['enabled']))
try:
    sec = session.sql("SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('GTMAGENTS_MCP_OAUTH') AS s").collect()[0]['S']
    checks.append(('OAuth secrets retrievable', 'OAUTH_CLIENT_ID' in sec))
except Exception as e:
    checks.append(('OAuth secrets retrievable', False))
session.sql("USE ROLE SYSADMIN").collect()

# DEFAULT_ROLE / DEFAULT_WAREHOUSE set on the MCP user (warn-level: non-null)
me = session.sql("SELECT CURRENT_USER() AS u").collect()[0]['U']
session.sql("USE ROLE ACCOUNTADMIN").collect()
urow = session.sql(f"SHOW USERS LIKE '{me}'").collect()[0].as_dict()
session.sql("USE ROLE SYSADMIN").collect()
checks.append(('MCP user has DEFAULT_ROLE', bool(urow.get('default_role'))))
checks.append(('MCP user has DEFAULT_WAREHOUSE', bool(urow.get('default_warehouse'))))

# Latency/cost logging wrote at least one MCP row
mcp_rows = session.sql("SELECT COUNT(*) AS n FROM REQUEST_LOG WHERE source='MCP'").collect()[0]['N']
checks.append(('>=1 MCP row in REQUEST_LOG', mcp_rows >= 1))

print('CHECKPOINT A')
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
# DEFAULT_ROLE/WH are warn-level (needed only for the live Claude connect), not hard blockers
hard = [ok for name, ok in checks if 'DEFAULT_' not in name]
print()
print('RESULT:', 'PASS — proceed to Part B' if all(hard) else 'FAIL — fix above before Part B')
if not all(ok for _, ok in checks):
    print('NOTE: set DEFAULT_ROLE/DEFAULT_WAREHOUSE before the live Claude connect (Section 4).')

---
## Summary

The BEFORE state is wired: a Snowflake-managed MCP server exposes the Part 0 tools, OAuth is configured, and
a latency/cost baseline is logged. Claude drives these tools as an **external brain** — powerful, but every
call pays a network + planning round-trip, and governance/observability live outside Snowflake. Next,
**`gtm-03-after-agents`** moves the brain **into the data plane** with a multi-agent Cortex Agents architecture.